In [ ]:
import streamlit as st
import pickle
import numpy as np
import pandas as pd
import warnings

warnings.filterwarnings('ignore')

# --- INITIALIZATION ---
st.set_page_config(page_title="IPL Pro Predictor v4", page_icon="🏏", layout="wide")

@st.cache_resource
def load_ipl_resources():
    # We load the dictionary containing the model AND the trained columns
    with open('ipl_model_v4.pkl', 'rb') as f:
        data = pickle.load(f)
    return data['model'], data['columns']

# Load model and the exact column list used during training
model, model_columns = load_ipl_resources()

# --- SIDEBAR MATCH CONTROL ---
st.sidebar.header("🏟️ Match Setup")
teams = [
    'Chennai Super Kings', 'Delhi Capitals', 'Gujarat Titans',
    'Kolkata Knight Riders', 'Lucknow Super Giants', 'Mumbai Indians',
    'Punjab Kings', 'Rajasthan Royals', 'Royal Challengers Bangalore',
    'Sunrisers Hyderabad'
]
cities = [
    'Mumbai', 'Chennai', 'Delhi', 'Kolkata', 'Hyderabad', 
    'Bangalore', 'Ahmedabad', 'Jaipur', 'Lucknow', 'Chandigarh'
]

batting_team = st.sidebar.selectbox("Batting Team (Chasing)", sorted(teams))
bowling_team = st.sidebar.selectbox("Bowling Team", sorted([t for t in teams if t != batting_team]))
city = st.sidebar.selectbox("Host City", sorted(cities))
target = st.sidebar.number_input("Target Score", 1, 300, 180)

# --- MAIN DASHBOARD ---
st.title("🏏 IPL Win Probability Predictor")
st.markdown("#### *Powered by Ball-by-Ball Historical Data (2008-2025)*")

col1, col2 = st.columns([2, 1])

with col1:
    st.subheader("📊 Match Situation")
    c1, c2, c3 = st.columns(3)
    with c1:
        runs_total = st.number_input("Current Score", 0, target + 10, 100)
    with c2:
        wickets_lost = st.number_input("Wickets Down", 0, 10, 3)
    with c3:
        over = st.number_input("Overs Completed", 0, 19, 10)

    # Calculation Logic
    runs_required = target - runs_total
    balls_remaining = 120 - (over * 6)
    wickets_remaining = 10 - wickets_lost
    rrr = round(runs_required / (balls_remaining / 6), 2) if balls_remaining > 0 else 99
    crr = round(runs_total / (over if over > 0 else 1), 2)

with col2:
    st.subheader("📈 Live Metrics")
    st.metric("Runs Needed", f"{runs_required} off {balls_remaining} balls")
    st.metric("Required Rate", f"{rrr}")

# --- THE PREDICTION ENGINE ---
def get_prediction():
    # 1. Math Overrides
    if runs_required <= 0: return 100.0
    if wickets_remaining == 0 or (balls_remaining <= 0 and runs_required > 0): return 0.0
    if balls_remaining > 0:
        max_theoretical_runs = balls_remaining * 6 
        if runs_required > max_theoretical_runs:
            return 0.0

    # 2. MATCH THE TRAINING COLUMNS (One-Hot Encoding Fix)
    # Create a DataFrame with 1 row and all columns as 0
    input_df = pd.DataFrame(np.zeros((1, len(model_columns))), columns=model_columns)

    # Fill numerical features
    input_df['runs_left'] = runs_required
    input_df['balls_left'] = balls_remaining
    input_df['wickets_remaining'] = wickets_remaining
    input_df['runs_target'] = target
    input_df['rrr'] = rrr
    input_df['crr'] = crr

    # Set categorical flags to 1
    if f'batting_team_{batting_team}' in model_columns:
        input_df[f'batting_team_{batting_team}'] = 1
    if f'bowling_team_{bowling_team}' in model_columns:
        input_df[f'bowling_team_{bowling_team}'] = 1
    if f'city_{city}' in model_columns:
        input_df[f'city_{city}'] = 1

    try:
        # Get actual probability from the Random Forest
        prob = model.predict_proba(input_df)[0]
        win_p = prob[1] * 100
        
        # 3. Dynamic Realism Filter
        if rrr > 25: win_p = min(win_p, 1.0)
        elif rrr > 15: win_p = win_p * (1 - ((rrr-15)/15))
        
        return round(max(0.1, min(99.9, win_p)), 1)
    except:
        return 50.0

if st.button("🔮 ANALYZE WIN CHANCE", use_container_width=True):
    win = get_prediction()
    loss = round(100 - win, 1)
    
    st.markdown("---")
    res1, res2 = st.columns(2)
    with res1:
        st.subheader(f"🔥 {batting_team}")
        st.title(f"{win}%")
        st.progress(int(win))
    with res2:
        st.subheader(f"🛡️ {bowling_team}")
        st.title(f"{loss}%")
        st.progress(int(loss))

st.markdown("---")
st.caption("Deepak's Professional Predictor | Multi-Feature Accuracy Mode")



#############################################################################################################################


